In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from src.annaive import v_func

In [2]:
Re = 40
nu = 1 / Re
l = 1 / (2 * nu) - np.sqrt(1 / (4 * nu ** 2) + 4 * np.pi ** 2)

In [12]:
def loss_function(coords, u, v, p):
    # First derivatives
    grad_u = torch.autograd.grad(u, coords, torch.ones_like(u), create_graph=True)[0]
    du_x = grad_u[:, 0]
    du_y = grad_u[:, 1]

    grad_v = torch.autograd.grad(v, coords, torch.ones_like(v), create_graph=True)[0]
    dv_x = grad_v[:, 0]
    dv_y = grad_v[:, 1]

    grad_p = torch.autograd.grad(p, coords, torch.ones_like(p), create_graph=True)[0]
    dp_x = grad_p[:, 0]
    dp_y = grad_p[:, 1]

    # Second derivatives
    du_xx = torch.autograd.grad(du_x, coords, torch.ones_like(du_x), create_graph=True)[0][:, 0]
    du_yy = torch.autograd.grad(du_y, coords, torch.ones_like(du_y), create_graph=True)[0][:, 1]

    dv_xx = torch.autograd.grad(dv_x, coords, torch.ones_like(dv_x), create_graph=True)[0][:, 0]
    dv_yy = torch.autograd.grad(dv_y, coords, torch.ones_like(dv_y), create_graph=True)[0][:, 1]

    ns_x = u * du_x + v * du_y + dp_x - nu * (du_xx + du_yy)
    ns_y = u * dv_x + v * dv_y + dp_y - nu * (dv_xx + dv_yy)
    continuity = du_x + dv_y

    return ns_x, ns_y, continuity

def u_func(x, y):
    return 1 - np.exp(l * x) * np.cos(2 * np.pi * y)

def v_func(x):
    return (l / (2 * np.pi)) * np.exp(l * x) * np.sin(2 * np.pi * x)

def p_func(x):
    return 1 / 2 * (1 - np.exp(2 * l * x))

In [4]:
PINNModel = nn.Sequential(
    nn.Linear(2, 50),
    nn.Tanh(),
    nn.Linear(50, 50),
    nn.Tanh(),
    nn.Linear(50, 50),
    nn.Tanh(),
    nn.Linear(50, 50),
    nn.Tanh(),
    nn.Linear(50, 50),
    nn.Tanh(),
    nn.Linear(50, 3)
).double()

numepochs = 15000
adam_optimizer = torch.optim.Adam(PINNModel.parameters(), lr=0.001)

mse = nn.MSELoss()

outside_points = torch.tensor(
    [[0, i] for i in np.linspace(0, 1, 50)] +
    [[1, i] for i in np.linspace(0, 1, 50)] +
    [[i, 0] for i in np.linspace(0, 1, 50)] +
    [[i, 1] for i in np.linspace(0, 1, 50)]
).double().requires_grad_(True)

inside_points = torch.tensor(
    np.random.rand(5000, 2)
).double().requires_grad_(True)

def train_model():
  losses = torch.zeros(numepochs)
  for i in range(numepochs):
    adam_optimizer.zero_grad()

    # Inside
    inside_values = PINNModel(inside_points)
    ns_x, ns_y, continuity = loss_function(inside_points, # Pass the full coordinates
                                           inside_values[:, 0],
                                           inside_values[:, 1],
                                           inside_values[:, 2]
                                          )

    ns_loss_x_inside, ns_loss_y_inside = mse(ns_x, torch.zeros_like(ns_x)), mse(ns_y, torch.zeros_like(ns_y))
    continuity_loss_inside = mse(continuity, torch.zeros_like(continuity))

    outside_values = PINNModel(outside_points)

    u_boundary_loss = mse(outside_values[:, 0], torch.zeros_like(outside_values[:, 0]))

    loss = ns_loss_x_inside + ns_loss_y_inside + continuity_loss_inside + u_boundary_loss
    losses[i] = loss.item()
    loss.backward()
    adam_optimizer.step()
  return losses

In [5]:
losses = train_model()

In [1]:
plt.plot(losses)

NameError: name 'plt' is not defined

In [10]:
torch.save(PINNModel.state_dict(), '../src/PINNModel_01.pt')

In [17]:
test_points = inside_points = torch.tensor(
    np.random.rand(5000, 2)
).double()

outside_results = PINNModel(outside_points)
inside_results = PINNModel(inside_points)
outside_true_results = [[(u_func(int(x), int(y)), v_func(int(x)), p_func(int(x))) for x in outside_points[:, 0]] for y in outside_points[:, 1]]
inside_true_results = [[(u_func(int(x), int(y)), v_func(int(x)), p_func(int(x))) for x in inside_points[:, 0]] for y in inside_points[:, 1]]

In [18]:
print(inside_results)
print(inside_true_results)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



coucu
